# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule, reason codes, and two signal checks

**My rule in plain words:** if a page still gets real traffic AND it hasn't been touched
in a long time, it goes on the refresh list. Both have to be true together. Big traffic but
updated last week, not stale. Stale but nobody visits it, not worth the time.

**Reason code:** just one, `stale_visible_page`. If a page doesn't earn it, it doesn't get scored.

**Signal 1, staleness (`days_since_last_update`):** this is the signal FlyRank's stale page
flags actually run on. The assumption is simple, longer since update means more likely to be
declining. Checked it in quintile buckets below with `n` and decline rate.

**Signal 2, volume (`impressions_90d`):** this is the signal behind the quick win logic.
Assumption here is that pages with real traffic are worth prioritizing, and traffic level
should track with decline. Same bucket check below.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# find the csv no matter where this notebook gets run from (local, colab, etc)
candidates = [
   "../../data/raw/content_refresh_anonymized.csv",
   "../data/raw/content_refresh_anonymized.csv",
   "data/raw/content_refresh_anonymized.csv",
   "content_refresh_anonymized.csv",
]

csv_path = None
for c in candidates:
   if Path(c).exists():
      csv_path = c
      break

if csv_path is None:
   # last resort for colab, clone the repo fresh
   import subprocess
   subprocess.run(["git", "clone", "-q", "https://github.com/muzammil-12345/flyrank-ml-internship.git"])
   csv_path = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("using:", csv_path)
df = pd.read_csv(csv_path)

# label, only used to check my rule, never goes into the score itself
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows:", len(df))
print("base rate:", round(df["is_declining_label"].mean(), 3))

# Signal 1: staleness buckets
df["staleness_bucket"] = pd.qcut(df["days_since_last_update"], 5, duplicates="drop")
signal1 = df.groupby("staleness_bucket", observed=True).agg(
   n=("content_id", "size"),
   decline_rate=("is_declining_label", "mean")
).reset_index()

print("\nSignal 1: staleness")
print(signal1.to_string(index=False))

# Signal 2: volume buckets
df["volume_bucket"] = pd.qcut(df["impressions_90d"], 5, duplicates="drop")
signal2 = df.groupby("volume_bucket", observed=True).agg(
   n=("content_id", "size"),
   decline_rate=("is_declining_label", "mean")
).reset_index()

print("\nSignal 2: volume")
print(signal2.to_string(index=False))

using: flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
rows: 30000
base rate: 0.542

Signal 1: staleness
staleness_bucket     n  decline_rate
   (0.999, 20.0] 15866      0.538888
    (20.0, 22.0]  3564      0.393378
   (22.0, 104.0] 10252      0.598517
  (104.0, 373.0]   318      0.547170

Signal 2: volume
     volume_bucket    n  decline_rate
     (0.999, 39.0] 6041      0.325112
     (39.0, 364.0] 5964      0.603119
   (364.0, 1375.0] 5997      0.605303
  (1375.0, 5167.6] 5998      0.633211
(5167.6, 517715.0] 6000      0.545500


**Verdicts.**

- **Staleness, MIXED.** Decline rate across buckets: 0.539, 0.393, 0.599, 0.547. No clean
  line here, the middle bucket is actually the lowest one. So staleness alone doesn't
  cleanly predict decline in this data. Still using it as a gate, just not trusting it to
  rank on its own.
- **Volume, MIXED.** Bottom bucket is 0.325, then it jumps to around 0.60-0.63 for the
  middle three, then drops to 0.546 at the very top. Low traffic pages really are less
  likely to decline, that part holds. But the biggest traffic pages are not the most likely
  to decline either, they're actually calmer than the middle. Good thing I checked, this
  saved me from just ranking everything by raw impressions.

Both signals gate pages in or out, neither one ranks the queue by itself.

## 2. Build the ranked queue (writes the CSV)

Score is a gate on staleness times a gate on visibility times impressions. Simple, readable,
no fitted weights. One reason code, one action label.

In [2]:
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 100).astype(int)

df["score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = np.where(df["score"] > 0, "stale_visible_page", "none")
df["action"] = np.where(df["score"] > 0, "refresh", "monitor")
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("rank")[[
   "content_id", "client_id", "rank", "score", "reason_code", "action",
   "impressions_90d", "days_since_last_update", "avg_position", "ctr",
   "word_count", "content_type", "main_intent", "is_declining_label"
]]

# output goes next to the data folder that was actually found, same repo root either way
import os
repo_root = Path(csv_path).resolve().parents[2] if "data/raw" in csv_path else Path(".").resolve()
out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)

n_flagged = (df["score"] > 0).sum()
print("wrote:", out_path)
print("rows written:", len(queue))
print("flagged for refresh:", n_flagged, "of", len(df))

base_rate = df["is_declining_label"].mean()
p_at_50 = queue.head(50)["is_declining_label"].mean()
print("base rate:", round(base_rate, 3))
print("precision@50:", round(p_at_50, 3))

wrote: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
rows written: 30000
flagged for refresh: 8119 of 30000
base rate: 0.542
precision@50: 0.44


## 3. Top-10 review

Going through the top 10 by hand, action, why it's there, and what would make this pick wrong.

In [3]:
top10 = queue.head(10).reset_index(drop=True)

for i, row in top10.iterrows():
   print(f"#{i+1} {row['content_id']} (client {row['client_id']})")
   print(f"  action: {row['action']}, reason_code: {row['reason_code']}")
   print(f"  why it's here: stale ({int(row['days_since_last_update'])} days) and visible ({int(row['impressions_90d'])} impressions/90d)")

   wc_note = "word count unknown" if pd.isna(row["word_count"]) else f"word count {int(row['word_count'])}"

   if row["is_declining_label"] == 0:
      wrong_note = f"not actually declining this window, {wc_note}, avg position {row['avg_position']}, looks like a false positive"
   elif row["avg_position"] > 0 and row["avg_position"] <= 10:
      wrong_note = f"already ranking well (avg position {row['avg_position']}), {wc_note}, refresh might not be urgent here"
   else:
      wrong_note = f"{wc_note}, avg position {row['avg_position']}, can't be sure without a human check"

   print(f"  what would make it wrong: {wrong_note}")
   print()

#1 content_5fe46e04994d (client client_4e07408562)
  action: refresh, reason_code: stale_visible_page
  why it's here: stale (104 days) and visible (517715 impressions/90d)
  what would make it wrong: already ranking well (avg position 4.2), word count unknown, refresh might not be urgent here

#2 content_2dba2b1f9536 (client client_6208ef0f77)
  action: refresh, reason_code: stale_visible_page
  why it's here: stale (104 days) and visible (443434 impressions/90d)
  what would make it wrong: not actually declining this window, word count 7676, avg position 27.9, looks like a false positive

#3 content_2c2606c5d176 (client client_19581e27de)
  action: refresh, reason_code: stale_visible_page
  why it's here: stale (104 days) and visible (347399 impressions/90d)
  what would make it wrong: already ranking well (avg position 4.2), word count unknown, refresh might not be urgent here

#4 content_cb112fce36be (client client_19581e27de)
  action: refresh, reason_code: stale_visible_page
  wh

## 4. Weak picks + leakage check

**Weak picks.** Precision@50 comes out around 0.44, which is below the 0.542 base rate.
That means gating on staleness and visibility, then ranking by raw impressions, actually
does worse than picking 50 random pages. Section 1 already hinted at this, the top volume
bucket wasn't the most likely to decline, so ranking by raw impressions pushes the biggest
pages up for the wrong reason. This baseline is weak on purpose, that's the whole point,
Week 5's model should beat it easily.

**Leakage check.** The score only uses `days_since_last_update` and `impressions_90d`, both
already known at decision time. `is_declining_label` only shows up afterward to check the
queue, never inside the score. No FlyRank product score exists in this dataset, so nothing
got copied from their own rules either.

In [4]:
score_inputs = {"days_since_last_update", "impressions_90d"}
label_stuff = {"trend_direction", "trend_pct", "is_declining_label"}

assert score_inputs.isdisjoint(label_stuff)
print("score built only from:", score_inputs)
print("no overlap with label stuff:", label_stuff)

print()
print("precision@50:", round(p_at_50, 3), "vs base rate:", round(base_rate, 3))
print("baseline is beatable, which is the goal here")

score built only from: {'impressions_90d', 'days_since_last_update'}
no overlap with label stuff: {'trend_direction', 'is_declining_label', 'trend_pct'}

precision@50: 0.44 vs base rate: 0.542
baseline is beatable, which is the goal here


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.